In [ ]:
# MOSPOLI LMS AI Navigation Colab - setup/start
# Run this in a GPU runtime. It installs dependencies, downloads Qwen3 GGUF Q5_K_M,
# starts llama.cpp, ai-navigation-service, Vite frontend, and Cloudflare tunnel.
import json
import os
import re
import shlex
import shutil
import socket
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path

REPO_URL = os.environ.get('REPO_URL', 'https://github.com/GODIMONGO/MOSPOLI_LMS')
REPO_DIR = Path(os.environ.get('REPO_DIR', '/content/MOSPOLI_LMS'))
MODEL_DIR = Path(os.environ.get('MODEL_DIR', str(REPO_DIR / 'models')))
MODEL_FILE = os.environ.get('MODEL_FILE', 'qwen3-embedding-4b-q5_k_m.gguf')
MODEL_PATH = Path(os.environ.get('MODEL_PATH', str(MODEL_DIR / MODEL_FILE)))
MODEL_URL = os.environ.get('MODEL_URL', 'https://huggingface.co/Qwen/Qwen3-Embedding-4B-GGUF/resolve/main/Qwen3-Embedding-4B-Q5_K_M.gguf?download=true')
LLAMA_DIR = Path(os.environ.get('LLAMA_DIR', '/content/llama.cpp'))
LLAMA_SERVER = Path(os.environ.get('LLAMA_SERVER', str(LLAMA_DIR / 'build/bin/llama-server')))
LLAMA_PORT = int(os.environ.get('LLAMA_PORT', '8080'))
LLAMA_HEALTH_TIMEOUT = int(os.environ.get('LLAMA_HEALTH_TIMEOUT', '900'))
AI_PORT = int(os.environ.get('AI_PORT', '3001'))
FRONTEND_PORT = int(os.environ.get('FRONTEND_PORT', '5173'))
LOG_DIR = Path(os.environ.get('LOG_DIR', str(REPO_DIR / '.colab/logs')))

def run(args, cwd=None, env=None, check=True):
    print('+', ' '.join(str(x) for x in args), flush=True)
    return subprocess.run([str(x) for x in args], cwd=cwd, env=env, check=check)

def run_capture(args, cwd=None, check=False):
    return subprocess.run([str(x) for x in args], cwd=cwd, check=check, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT).stdout

def command_exists(name):
    return shutil.which(name) is not None

def node_is_new_enough():
    if not command_exists('node'):
        return False
    output = run_capture(['node', '--version']).strip().lstrip('v')
    major, minor, *_ = [int(part) for part in output.split('.')]
    return major > 20 or (major == 20 and minor >= 19)

def http_status(url, timeout=2):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            return response.status
    except urllib.error.HTTPError as error:
        return error.code
    except Exception:
        return None

def tail_log(log_file, lines=80):
    if not log_file or not Path(log_file).exists():
        return ''
    return '\n'.join(Path(log_file).read_text(errors='ignore').splitlines()[-lines:])

def wait_http(urls, seconds, name, process=None, log_file=None):
    if isinstance(urls, (str, bytes)):
        urls = [urls]
    deadline = time.time() + seconds
    last_report = 0
    while time.time() < deadline:
        statuses = {url: http_status(url) for url in urls}
        if any(status is not None and 200 <= status < 400 for status in statuses.values()):
            return
        if process is not None and process.poll() is not None:
            print(f'{name} exited with code {process.returncode}')
            recent = tail_log(log_file)
            if recent:
                print(f'--- recent {log_file}')
                print(recent)
            raise RuntimeError(f'{name} exited before becoming ready: {statuses}')
        if time.time() - last_report > 30:
            print(f'waiting for {name}: {statuses}', flush=True)
            last_report = time.time()
        time.sleep(2)
    recent = tail_log(log_file)
    if recent:
        print(f'--- recent {log_file}')
        print(recent)
    raise RuntimeError(f'{name} did not become ready: {urls}')

def start_background(args, log_file, cwd=None, env=None):
    log_file.parent.mkdir(parents=True, exist_ok=True)
    handle = log_file.open('ab')
    print('+ background', ' '.join(str(x) for x in args), '>', log_file, flush=True)
    return subprocess.Popen([str(x) for x in args], cwd=cwd, env=env, stdout=handle, stderr=subprocess.STDOUT, start_new_session=True)

print('== System packages ==')
run(['apt-get', 'update', '-y'])
run(['apt-get', 'install', '-y', 'git', 'curl', 'wget', 'ca-certificates', 'cmake', 'build-essential', 'libcurl4-openssl-dev', 'jq', 'lsof'])

if not node_is_new_enough():
    print('== Installing Node.js 22 ==')
    run(['curl', '-fsSL', 'https://deb.nodesource.com/setup_22.x', '-o', '/tmp/nodesource_setup.sh'])
    run(['bash', '/tmp/nodesource_setup.sh'])
    run(['apt-get', 'install', '-y', 'nodejs'])
print(run_capture(['node', '--version']).strip())
print(run_capture(['npm', '--version']).strip())

print('== Project ==')
if (REPO_DIR / '.git').is_dir():
    run(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
else:
    run(['git', 'clone', REPO_URL, REPO_DIR])
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print('== npm dependencies and builds ==')
run(['npm', 'ci' if (REPO_DIR / 'package-lock.json').exists() else 'install'], cwd=REPO_DIR)
service_dir = REPO_DIR / 'ai-navigation-service'
run(['npm', 'ci' if (service_dir / 'package-lock.json').exists() else 'install'], cwd=service_dir)
run(['npm', 'run', 'build'], cwd=REPO_DIR)
run(['npm', 'run', 'build'], cwd=service_dir)

print('== Qwen3 embedding model ==')
if not MODEL_PATH.exists():
    run(['curl', '-L', '--fail', '--retry', '5', '--retry-delay', '5', '-C', '-', '-o', MODEL_PATH, MODEL_URL])
else:
    print(f'Model already exists: {MODEL_PATH}')
run(['ls', '-lh', MODEL_PATH])

print('== llama.cpp ==')
cuda_available = command_exists('nvidia-smi') and run(['nvidia-smi'], check=False).returncode == 0
if not LLAMA_SERVER.exists():
    if (LLAMA_DIR / '.git').is_dir():
        run(['git', '-C', LLAMA_DIR, 'pull', '--ff-only'])
    else:
        run(['git', 'clone', 'https://github.com/ggml-org/llama.cpp', LLAMA_DIR])
    cmake_args = ['cmake', '-S', LLAMA_DIR, '-B', LLAMA_DIR / 'build', '-DCMAKE_BUILD_TYPE=Release']
    if cuda_available:
        cmake_args.append('-DGGML_CUDA=ON')
    else:
        print('No CUDA GPU detected; building CPU llama.cpp. For Qwen3-Embedding-4B, Colab GPU is strongly recommended.')
    run(cmake_args)
    run(['cmake', '--build', LLAMA_DIR / 'build', '--config', 'Release', '-j', str(os.cpu_count() or 2), '--target', 'llama-server'])
run([LLAMA_SERVER, '--version'], check=False)

print('== cloudflared tunnel client ==')
if not command_exists('cloudflared'):
    run(['wget', '-q', '-O', '/tmp/cloudflared-linux-amd64.deb', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'])
    run(['dpkg', '-i', '/tmp/cloudflared-linux-amd64.deb'])
run(['cloudflared', '--version'])

print('== Stop old Colab processes ==')
for pattern in ['llama-server', 'ai-navigation-service/dist/server.js', 'vite', 'cloudflared tunnel']:
    run(['pkill', '-f', pattern], check=False)
time.sleep(2)

print('== Start llama.cpp embedding server ==')
llama_log = LOG_DIR / 'llama-server.log'
llama_args = [
    LLAMA_SERVER,
    '-m', MODEL_PATH,
    '--embedding',
    '-ub', os.environ.get('LLAMA_UBATCH', '1024'),
    '-b', os.environ.get('LLAMA_BATCH', '1024'),
    '--host', '0.0.0.0',
    '--port', str(LLAMA_PORT),
]
if cuda_available:
    llama_args.extend(['-ngl', os.environ.get('LLAMA_GPU_LAYERS', '999')])
if os.environ.get('LLAMA_EXTRA_ARGS'):
    llama_args.extend(shlex.split(os.environ['LLAMA_EXTRA_ARGS']))
llama_process = start_background(llama_args, llama_log)
wait_http([
    f'http://127.0.0.1:{LLAMA_PORT}/health',
    f'http://127.0.0.1:{LLAMA_PORT}/v1/health',
    f'http://127.0.0.1:{LLAMA_PORT}/v1/models',
], LLAMA_HEALTH_TIMEOUT, 'llama.cpp', process=llama_process, log_file=llama_log)

print('== Start ai-navigation-service ==')
service_env = os.environ.copy()
service_env.update({
    'PORT': str(AI_PORT),
    'EMBEDDING_BASE_URL': f'http://127.0.0.1:{LLAMA_PORT}',
    'EMBEDDING_MODEL': 'qwen3-embedding-q5-k-m',
    'EMBEDDING_MOCK': 'false',
    'ALLOW_EMBEDDING_FALLBACK': 'false',
    'VECTOR_WEIGHT': '0.7',
    'KEYWORD_WEIGHT': '0.3',
    'PRIORITY_WEIGHT': '0.05',
    'T_REDIRECT': '0.82',
    'T_GAP': '0.08',
    'T_SUGGEST': '0.62',
    'SUGGESTIONS_COUNT': '5',
})
start_background(['node', 'ai-navigation-service/dist/server.js'], LOG_DIR / 'ai-navigation-service.log', cwd=REPO_DIR, env=service_env)
wait_http(f'http://127.0.0.1:{AI_PORT}/health', 360, 'ai-navigation-service')
with urllib.request.urlopen(f'http://127.0.0.1:{AI_PORT}/health') as response:
    print(json.dumps(json.load(response), ensure_ascii=False, indent=2))

print('== Start Vite frontend ==')
frontend_env = os.environ.copy()
frontend_env.update({
    'AI_NAVIGATION_UPSTREAM': f'http://127.0.0.1:{AI_PORT}',
    'VITE_AI_NAVIGATION_URL': '',
})
start_background(['npm', 'run', 'dev', '--', '--host', '0.0.0.0', '--port', str(FRONTEND_PORT)], LOG_DIR / 'frontend.log', cwd=REPO_DIR, env=frontend_env)
wait_http(f'http://127.0.0.1:{FRONTEND_PORT}/', 180, 'frontend')

print('== Start Cloudflare quick tunnel ==')
start_background(['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{FRONTEND_PORT}', '--no-autoupdate'], LOG_DIR / 'cloudflared.log')
tunnel_url = ''
cloudflared_log = LOG_DIR / 'cloudflared.log'
for _ in range(90):
    if cloudflared_log.exists():
        match = re.findall(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com', cloudflared_log.read_text(errors='ignore'))
        if match:
            tunnel_url = match[-1]
            break
    time.sleep(1)

print('\nMOSPOLI LMS AI Navigation is running.')
print(f'Frontend local: http://127.0.0.1:{FRONTEND_PORT}')
print(f'AI API local:    http://127.0.0.1:{AI_PORT}')
print(f'llama.cpp:       http://127.0.0.1:{LLAMA_PORT}')
print(f'Logs:            {LOG_DIR}')
if tunnel_url:
    print(f'Public URL:      {tunnel_url}')
else:
    print('Public URL was not detected yet. Run the check/logs cell and inspect cloudflared.log.')


In [ ]:
import json
import os
import re
import subprocess
import urllib.request
from pathlib import Path

REPO_DIR = Path(os.environ.get('REPO_DIR', '/content/MOSPOLI_LMS'))
AI_PORT = int(os.environ.get('AI_PORT', '3001'))
FRONTEND_PORT = int(os.environ.get('FRONTEND_PORT', '5173'))
LOG_DIR = Path(os.environ.get('LOG_DIR', str(REPO_DIR / '.colab/logs')))

def out(args):
    print(subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT).stdout)

def get_json(url):
    with urllib.request.urlopen(url, timeout=30) as response:
        return json.load(response)

print('== Processes ==')
out(['bash', '-lc', "ps -ef | grep -E 'llama-server|ai-navigation-service/dist/server.js|vite|cloudflared tunnel' | grep -v grep || true"])

print('== Ports ==')
out(['bash', '-lc', "lsof -iTCP -sTCP:LISTEN -P | grep -E ':(8080|3001|5173)' || true"])

print('== Health ==')
print(json.dumps(get_json(f'http://127.0.0.1:{AI_PORT}/health'), ensure_ascii=False, indent=2))

print('== Search smoke test ==')
request = urllib.request.Request(
    f'http://127.0.0.1:{FRONTEND_PORT}/api/navigation-search',
    data=json.dumps({'query': 'регистрация', 'locale': 'ru'}).encode('utf-8'),
    headers={'Content-Type': 'application/json'},
    method='POST',
)
with urllib.request.urlopen(request, timeout=120) as response:
    print(json.dumps(json.load(response), ensure_ascii=False, indent=2))

print('== Tunnel URL ==')
cloudflared_log = LOG_DIR / 'cloudflared.log'
if cloudflared_log.exists():
    matches = re.findall(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com', cloudflared_log.read_text(errors='ignore'))
    print(matches[-1] if matches else '')

print('== Recent logs ==')
for name in ['llama-server', 'ai-navigation-service', 'frontend', 'cloudflared']:
    print(f'--- {name}.log')
    path = LOG_DIR / f'{name}.log'
    if path.exists():
        print('\n'.join(path.read_text(errors='ignore').splitlines()[-40:]))
    else:
        print('missing')
